In [1]:
import os
from main import *
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
import datetime
import calendar
import holidays

In [11]:
pst = pytz.timezone("US/Pacific")
pst.localize(datetime.datetime(2025, 12, 10)).dst()

datetime.timedelta(0)

In [2]:
def add_sun_moon_info(df: pd.DataFrame) -> None:
    moon_illumination, sunrise, sunset = [], [], []
    for t in df['low_tide_time']:
        month, day, year = t.month, t.day, t.year
        moon_illumination.append(get_moon_illumination_on_day(year, int(month), int(day)))
        srise, sset = get_sun_times_on_day(year, int(month), int(day))
        sunrise.append(srise)
        sunset.append(sset)
    df['moon_illumination'] = moon_illumination
    df['sunrise'] = sunrise
    df['sunset'] = sunset

In [3]:
def add_date_label(df: pd.DataFrame) -> None:
    # ['night', 'workday', 'holiday', 'weekend']
    
    # checks if lowtide is within 1 hour of sunrise or sunset
    def is_night_low_tide(df:pd.DataFrame) -> bool:
        if abs(df['low_tide_time'] - df['sunrise']).seconds / 3600 < 1.5:
            return False
        if abs(df['low_tide_time'] - df['sunset']).seconds / 3600 < 1.5:
            return False
        if df['low_tide_time'] > df['sunrise'] and df['low_tide_time'] < df['sunset']:
            return False
        return True
    
    us_holidays = holidays.US(years=df.iloc[0]['low_tide_time'].year)
        
    label = []
    for _, row in df.iterrows():
        # check if low tide is at night
        if is_night_low_tide(row):
            label.append('night')
            continue
            
        # check if date is on a weekend
        if row['low_tide_time'].isoweekday() in [6, 7]:
            label.append('weekend')
            continue
        
        # check if date is on a US holiday
        if row['low_tide_time'] in us_holidays:
            label.append('holiday')
            continue
        
        label.append('workday')
        
    df['label'] = label

In [4]:
def collapse_to_datetime(df:pd.DataFrame, year:int) -> None:
    dates = []
    for row in range(df.shape[0]):
        df_row = df.iloc[row]
        time = datetime.datetime.strptime(f"{df_row['date']}/{year} {df_row['time']}", "%m/%d/%Y %I:%M %p")
        dates.append(pd.Timestamp(time).tz_localize('US/PACIFIC'))
    df['low_tide_time'] = dates

In [5]:

# setup meta vars for url
year = '2025'
start_date = f'{year}0101'
end_date = f'{year}1231'
station_id = '9414131' # half moon bay pillar point
url = f'https://tidesandcurrents.noaa.gov/cgi-bin/predictiondownload.cgi?&stnid={station_id}&threshold=&thresholdDirection=&bdate={start_date}&edate={end_date}&units=standard&timezone=LST/LDT&datum=MLLW&interval=hilo&clock=12hour&type=txt&annual=false'

# save the file downloaded from NOAA to tides.txt
filename = 'tides.txt' 
filepath = os.getcwd()

txt_filename = '1.txt'
csv_filename = '1.csv'

# check if the file is already there
if os.path.exists(os.path.join(filepath, filename)):
    print(f"File {filename} already exists.  Skipping download.")
else:
    # download the data from NOAA
    print('Downloading data from NOAA...')
    filename, _ = urllib.request.urlretrieve(url, filename)
    print(f"Data saved to {filename}")

#  remove the first empty row
first_empty_row = find_first_empty_row(os.path.join(filepath, filename))
remove_first_x_rows(filepath, filename, txt_filename ,first_empty_row)
print(f"First {first_empty_row } rows removed from {filename}.")

if first_empty_row:
    print(f"The first empty row is: {first_empty_row}")
else:
    print("No empty rows found.")

txt_to_csv(os.path.join(filepath, txt_filename), os.path.join(filepath, csv_filename), delimiter='\t')

input_file = os.path.join(filepath, csv_filename) # name the tide csv to 1.csv
output_file = os.path.join(filepath, 'dates_low_tides.csv')
threshold = -0.6  # Replace with your desired tide level, in ft

print("File path:", filepath)
if os.access(filepath, os.R_OK):
    print("File is readable")
else:
    print("File is not readable")
print ("Only dates with tide < ", threshold , " are left")
    
df = filter_dates(input_file, output_file, threshold)
df = df.rename(columns={
    0: 'date',
    1: 'day_of_week',
    2: 'time',
    3: 'low_tide'
})

collapse_to_datetime(df, year)
del df['date']
del df['day_of_week']
del df['time']

    # Save the filtered DataFrame to a new CSV file
df.to_csv(output_file, index=False) 

File tides.txt already exists.  Skipping download.
First 13 rows removed from tides.txt.
The first empty row is: 13
Converted /Users/ericdong/tidepool/1.txt to /Users/ericdong/tidepool/1.csv
File path: /Users/ericdong/tidepool
File is readable
Only dates with tide <  -0.6  are left
Total rows: 1411
Total columns: 5
          0    1         2     3  4
0     01/01  Wed  12:23 AM  4.32  H
1     01/01  Wed  04:34 AM  3.07  L
2     01/01  Wed  10:32 AM  6.32  H
3     01/01  Wed  06:02 PM -1.02  L
4     01/02  Thu  12:58 AM  4.44  H
...     ...  ...       ...   ... ..
1406  12/30  Tue  08:17 PM  3.79  H
1407  12/31  Wed  12:25 AM  2.64  L
1408  12/31  Wed  06:55 AM  6.75  H
1409  12/31  Wed  02:32 PM -1.13  L
1410  12/31  Wed  09:20 PM  4.07  H

[1411 rows x 5 columns]
lowest tide in 2025 is  -2.06 ft


In [9]:
add_sun_moon_info(df)
add_date_label(df)

In [11]:
df[df['label'] == 'night']

,low_tide,low_tide_time,moon_illumination,sunrise,sunset,label
7,-0.80,2025-01-02 18:39:00-08:00,13.955942,2025-01-02 07:18:48-08:00,2025-01-02 16:55:24-08:00,night
449,-1.21,2025-04-27 05:03:00-07:00,5.540597,2025-04-27 07:12:12-07:00,2025-04-27 20:49:24-07:00,night
557,-1.27,2025-05-25 04:05:00-07:00,14.509347,2025-05-25 06:45:48-07:00,2025-05-25 21:13:24-07:00,night
561,-1.81,2025-05-26 04:53:00-07:00,7.686511,2025-05-26 06:45:12-07:00,2025-05-26 21:14:36-07:00,night
665,-0.85,2025-06-22 03:04:00-07:00,23.461850,2025-06-22 06:41:36-07:00,2025-06-22 21:27:48-07:00,night
669,-1.45,2025-06-23 03:56:00-07:00,16.638701,2025-06-23 06:41:36-07:00,2025-06-23 21:27:48-07:00,night
673,-1.80,2025-06-24 04:46:00-07:00,9.815552,2025-06-24 06:41:36-07:00,2025-06-24 21:28:24-07:00,night
731,-0.72,2025-07-09 04:59:00-07:00,92.377774,2025-07-09 06:48:48-07:00,2025-07-09 21:26:36-07:00,night
777,-0.80,2025-07-21 02:55:00-07:00,25.880626,2025-07-21 06:57:12-07:00,2025-07-21 21:20:36-07:00,night
781,-1.19,2025-07-22 03:49:00-07:00,19.068826,2025-07-22 06:57:48-07:00,2025-07-22 21:20:00-07:00,night


In [8]:
# Create figure with 2 subplots
rows = 4
cols = 3
month_names = calendar.month_name[1:]

color_lut = {
    'night'  : 'black', 
    'workday': 'red', 
    'holiday': 'orange', 
    'weekend': 'green'
}

fig = make_subplots(
    rows=rows, 
    cols=cols,
    subplot_titles=month_names,
    horizontal_spacing = 0.03,
    vertical_spacing=0.08
)

# traversing through each possible label for the legend
for label in df['label'].unique():
    for row in range(rows):
        for col in range(cols):
            month = row * cols + col + 1
            filter_df = df[(df['low_tide_time'].dt.month == month) & (df['label'] == label)]
            
            colors = [color_lut[label] for label in filter_df['label']]
            hovertext = [
                f"""
    Low tide time: {row['low_tide_time'].strftime('%a %m/%d/%Y %I:%M %p')}<br>
    Low tide: {row['low_tide']} ft<br>
    Sunrise time: {row['sunrise'].strftime('%a %m/%d/%Y %I:%M %p')}<br>
    Sunset time: {row['sunset'].strftime('%a %m/%d/%Y %I:%M %p')}<br>
    Moon Illumination: {row['moon_illumination']:.0f}%<br>
    Label: {row['label']}
                """
                for _, row in filter_df.iterrows()
            ]
            
            fig.add_trace(
                go.Bar(
                    x=filter_df['low_tide_time'].dt.day, 
                    y=filter_df['low_tide'],
                    name=label,
                    marker_color=colors,
                    hovertext=hovertext,
                    showlegend=(row==0) and (col==0) # we want a custom legend for the possible labels
                ),
                row=row+1, 
                col=col+1,
            )
            
for row in range(rows):
    for col in range(cols):
        month = row * cols + col + 1
        filter_df = df[df['low_tide_time'].dt.month == month]
        x_ticks = set([1] + list(filter_df['low_tide_time'].dt.day) + [calendar.monthrange(int(year), month)[1]])
        fig.update_xaxes(
            tickvals=list(x_ticks),
            range=[0, calendar.monthrange(int(year), month)[1]+1],
            row=row+1, 
            col=col+1
        )

fig.update_layout(
    height=1000, 
    width=2000, 
    title_text="Side By Side Subplots",
    # showlegend=False
)
fig.show()